# 1주차 — 텍스트 입력 파이프라인

개념 설명은 [`notes/01-input-pipeline.md`](../notes/01-input-pipeline.md)

GPU 불필요. 토크나이저 파일(수 MB)만 내려받고 모델 가중치는 쓰지 않습니다.

In [1]:
%pip install -q transformers tokenizers numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
from transformers import AutoConfig, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
text = "Hello World!"
print("vocab size:", len(tok))

/Users/kimtaeyeong/miniconda3/envs/nlp-study/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


vocab size: 50257


## 1. Tokenization

In [3]:
# 어휘에 있는 단어는 통째로, 없는 단어는 조각으로 쪼개진다
for w in ["Hello", "World", "tokenization", "lowest", "antidisestablishmentarianism"]:
    print(f"{w:30s} -> {tok.tokenize(w)}")

Hello                          -> ['Hello']
World                          -> ['World']
tokenization                   -> ['token', 'ization']
lowest                         -> ['low', 'est']
antidisestablishmentarianism   -> ['ant', 'idis', 'establishment', 'arian', 'ism']


### 1.1. BPE로 어휘 만들기

라이브러리 API 이름은 `train`이지만 신경망 학습이 아닙니다.
gradient도 파라미터도 없고 빈도만 세므로, 같은 코퍼스면 항상 같은 결과가 나옵니다.

In [4]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers


def build_bpe(words, vocab_size):
    """코퍼스에서 병합 규칙을 뽑아 어휘를 구축한다."""
    t = Tokenizer(models.BPE())
    t.pre_tokenizer = pre_tokenizers.Whitespace()
    t.train_from_iterator(
        [" ".join(words)],
        trainers.BpeTrainer(vocab_size=vocab_size, show_progress=False),
    )
    return t

In [5]:
# 글의 예제 코퍼스. ID 순서가 곧 병합 순서다
corpus = ["low"] * 5 + ["lower"] * 2 + ["newest"] * 6 + ["widest"] * 3
bpe = build_bpe(corpus, vocab_size=30)

merged = [t for t, _ in sorted(bpe.get_vocab().items(), key=lambda kv: kv[1]) if len(t) > 1]
print("병합 순서:", merged)

병합 순서: ['es', 'est', 'lo', 'low', 'ew', 'new', 'newest', 'dest', 'idest', 'widest', 'er', 'lower']


In [6]:
# lowest / slowest 는 코퍼스에 없던 단어
for w in ["low", "lowest", "slowest"]:
    mark = "학습됨" if w in corpus else "처음봄"
    print(f"{w:9s} {mark}  ->  {bpe.encode(w).tokens}")

low       학습됨  ->  ['low']
lowest    처음봄  ->  ['low', 'est']
slowest   처음봄  ->  ['s', 'low', 'est']


In [7]:
# 어휘를 키우면 문장당 토큰 수는 줄지만, 임베딩 행렬의 행이 그만큼 늘어난다
words = "the quick brown fox jumps over the lazy dog near the river bank".split() * 40
for size in [35, 45, 60, 80]:
    pieces = build_bpe(words, size).encode("jumping over riverbanks").tokens
    print(f"vocab {size:>3} | {len(pieces):>2}토큰 | {pieces}")

vocab  35 | 15토큰 | ['j', 'u', 'm', 'p', 'i', 'n', 'g', 'o', 'ver', 'r', 'i', 'ver', 'ban', 'k', 's']
vocab  45 | 12토큰 | ['ju', 'mp', 'i', 'n', 'g', 'o', 'ver', 'r', 'iver', 'ban', 'k', 's']
vocab  60 |  8토큰 | ['jump', 'i', 'n', 'g', 'over', 'river', 'bank', 's']
vocab  80 |  8토큰 | ['jump', 'i', 'n', 'g', 'over', 'river', 'bank', 's']


## 2. Converting Tokens to Input IDs

In [8]:
# Ġ 는 앞에 공백이 있었다는 표시
tokens = tok.tokenize(text)
ids = tok.convert_tokens_to_ids(tokens)

print(f"{text!r}\n  -> {tokens}\n  -> {ids}")
print("\n되돌리기:", repr(tok.decode(ids)))

'Hello World!'
  -> ['Hello', 'ĠWorld', '!']
  -> [15496, 2159, 0]

되돌리기: 'Hello World!'


## 3. Special Tokens

In [9]:
# bos/eos/unk 가 같은 토큰이고 pad 는 없다
for name in ["bos_token", "eos_token", "unk_token", "pad_token"]:
    print(f"{name:10s} = {getattr(tok, name)!r:18s} id = {getattr(tok, name + '_id')}")

bos_token  = '<|endoftext|>'    id = 50256
eos_token  = '<|endoftext|>'    id = 50256
unk_token  = '<|endoftext|>'    id = 50256
pad_token  = None               id = None


In [10]:
# 자동으로 붙지 않으므로 직접 넣어야 한다
print(tok(text)["input_ids"])
print(tok(text + tok.eos_token)["input_ids"])

[15496, 2159, 0]
[15496, 2159, 0, 50256]


## 4. Padding and Attention Mask

In [11]:
# pad_token 이 없으면 배치를 못 만든다
try:
    tok(["Hello World!", "Yes"], padding=True)
except ValueError as e:
    print("ValueError:", str(e)[:90], "...")

ValueError: Asking to pad but the tokenizer does not have a padding token. Please select a token to us ...


In [12]:
tok.pad_token = tok.eos_token
batch = tok(["Hello World!", "The dog bit him", "Yes"], padding=True)

for row, mask in zip(batch["input_ids"], batch["attention_mask"]):
    print(f"{str(row):32s} {mask}")

[15496, 2159, 0, 50256]          [1, 1, 1, 0]
[464, 3290, 1643, 683]           [1, 1, 1, 1]
[5297, 50256, 50256, 50256]      [1, 0, 0, 0]


In [13]:
# 마스크가 0인 자리는 softmax 전에 -inf 가 되어 가중치 0이 된다
scores = np.array([2.0, 1.0, 0.5, 3.0])  # 마지막이 <pad> 위치
mask = np.array([1, 1, 1, 0])


def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()


print("마스크 없이:", softmax(scores).round(3))
print("마스크 적용:", softmax(np.where(mask == 1, scores, -np.inf)).round(3))

마스크 없이: [0.232 0.085 0.052 0.631]
마스크 적용: [0.629 0.231 0.14  0.   ]


## 5. Embedding Lookup

In [14]:
# 가중치는 받지 않고 설정값만 읽는다
cfg = AutoConfig.from_pretrained("gpt2")
V, d = cfg.vocab_size, cfg.n_embd
print(f"V = {V:,}   d = {d}   임베딩 파라미터 = {V * d / 1e6:.1f}M")

V = 50,257   d = 768   임베딩 파라미터 = 38.6M


In [15]:
# 실제 GPT-2 가중치 대신 같은 모양의 랜덤 표로 룩업을 재현한다
rng = np.random.default_rng(0)
table = rng.normal(0, 0.02, size=(V, d)).astype(np.float32)

vectors = table[ids]  # 인덱싱 한 번이 곧 룩업
print(f"{ids}  ->  {vectors.shape}\n")
for i, t in zip(ids, tokens):
    print(f"{i:>6}번 행  {np.round(table[i][:4], 3)} ...  <- {t!r}")

[15496, 2159, 0]  ->  (3, 768)

 15496번 행  [-0.008  0.003  0.011 -0.016] ...  <- 'Hello'
  2159번 행  [-0.031 -0.034  0.019 -0.011] ...  <- 'ĠWorld'
     0번 행  [ 0.003 -0.003  0.013  0.002] ...  <- '!'


In [16]:
# one-hot 과 행렬을 곱한 결과와 같다
onehot = np.zeros((len(ids), V), dtype=np.float32)
onehot[np.arange(len(ids)), ids] = 1.0
print("인덱싱 == one-hot 곱:", np.allclose(onehot @ table, vectors))

인덱싱 == one-hot 곱: True


## 번외. 어휘에 토큰 추가하기

도메인 용어가 계속 잘게 쪼개질 때 어휘에 직접 넣을 수 있습니다.
새 번호는 기존 어휘 맨 뒤에 붙습니다.

In [17]:
# 추가 전에는 조각마다 번호가 붙고, 추가 후에는 번호 하나가 새로 배정된다
word = "myoglobin"
print(f"추가 전   {tok.tokenize(word)}\n          ids = {tok(word)['input_ids']}\n")

tok.add_tokens([word])
print(f"추가 후   {tok.tokenize(word)}\n          ids = {tok(word)['input_ids']}\n")
print(f"어휘 크기 {V:,} -> {len(tok):,}")

추가 전   ['my', 'oglobin']
          ids = [1820, 49835]

추가 후   ['myoglobin']
          ids = [50257]

어휘 크기 50,257 -> 50,258


In [18]:
# 새 번호는 표의 범위 밖이라 그대로는 못 꺼낸다
new_id = tok(word)["input_ids"][0]
try:
    table[new_id]
except IndexError as e:
    print(f"table.shape = {table.shape},  요청한 행 = {new_id}")
    print(f"IndexError: {e}")

table.shape = (50257, 768),  요청한 행 = 50257
IndexError: index 50257 is out of bounds for axis 0 with size 50257


In [19]:
# 표에 행을 추가하면 해결된다. 실제로는 resize_token_embeddings 가 이 일을 한다
table = np.vstack([table, rng.normal(0, 0.02, size=(len(tok) - V, d)).astype(np.float32)])
print(f"table.shape = {table.shape}")
print(f"{new_id}번 행  {np.round(table[new_id][:4], 3)} ...  <- 랜덤 초기화 상태")

table.shape = (50258, 768)
50257번 행  [ 0.036  0.005  0.006 -0.015] ...  <- 랜덤 초기화 상태
